# jaxfne Computational Biophysics Tutorial

**Learning Goal:** Build and optimize a simulated cortical column using jaxfne and jbiophysic.

**Claim Level:** computational_scaffold (tutorial, not biological validation)

**Truth Mode:** truth_safe_unverified

**Time Estimate:** 15–20 minutes on CPU

---

## Section 0: Setup & Dependencies

Install and configure dependencies, confirm versions.

In [ ]:
import subprocess, sys
packages = ['numpy', 'scipy', 'pandas', 'matplotlib']
for pkg in packages:
    try:
        __import__(pkg)
        print(f'✓ {pkg} installed')
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('\nAll core dependencies ready.')

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt
from pathlib import Path
import json
from datetime import datetime

np.random.seed(42)
print('Imports successful.')

In [ ]:
figures_dir = Path('figures')
figures_dir.mkdir(exist_ok=True)

def savefig(name, fig=None, dpi=150):
    if fig is None:
        fig = plt.gcf()
    png_path = figures_dir / f'{name}.png'
    svg_path = figures_dir / f'{name}.svg'
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight')
    fig.savefig(svg_path, bbox_inches='tight')
    print(f'✓ {name} (PNG, SVG)')

def json_safe_dump(obj, path):
    '''Serialize with NaN/Inf -> None.'''
    def convert(o):
        if isinstance(o, float):
            return None if (np.isnan(o) or np.isinf(o)) else o
        elif isinstance(o, np.ndarray):
            return o.tolist()
        elif isinstance(o, dict):
            return {k: convert(v) for k, v in o.items()}
        elif isinstance(o, (list, tuple)):
            return [convert(v) for v in o]
        return o
    with open(path, 'w') as f:
        json.dump(convert(obj), f, indent=2)
    print(f'✓ {path}')

try:
    import importlib.metadata
    jaxfne_version = importlib.metadata.version('jaxfne')
except:
    jaxfne_version = 'unavailable'

print(f'\nVersions: NumPy {np.__version__}, jaxfne {jaxfne_version}')
print('Setup complete.')

## Section 1: Computational Models and Biophysics

Simulate a single Izhikevich neuron with step current input.

In [ ]:
class IzhikevichNeuron:
    def __init__(self, a=0.02, b=0.2, c=-65, d=8, v_init=-65, u_init=-13):
        self.a, self.b, self.c, self.d = a, b, c, d
        self.v, self.u = v_init, u_init
    
    def step(self, I_input, dt=0.1):
        dv = 0.04 * self.v**2 + 5 * self.v + 140 - self.u + I_input
        du = self.a * (self.b * self.v - self.u)
        self.v += dv * dt
        self.u += du * dt
        spike = self.v >= 30
        if spike:
            self.v = self.c
            self.u += self.d
        return self.v, spike

neuron = IzhikevichNeuron()
dt_ms, duration_ms = 0.1, 500
n_steps = int(duration_ms / dt_ms)
time_ms = np.arange(n_steps) * dt_ms

I_input = np.zeros(n_steps)
I_input[int(200/dt_ms):int(400/dt_ms)] = 10

v_trace, spike_times = np.zeros(n_steps), []
for i in range(n_steps):
    v, spike = neuron.step(I_input[i], dt_ms)
    v_trace[i] = v
    if spike:
        spike_times.append(time_ms[i])

print(f'Simulation: {len(spike_times)} spikes')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

ax = axes[0]
ax.plot(time_ms, v_trace, 'b-', linewidth=1)
ax.axhline(30, color='r', linestyle='--', alpha=0.5)
ax.set_ylabel('V (mV)')
ax.set_title('Single Neuron: Voltage Trace')
ax.grid(True, alpha=0.3)

ax = axes[1]
if spike_times:
    ax.vlines(spike_times, 0, 1, color='r', linewidth=2)
ax.set_ylim(-0.5, 1.5)
ax.set_ylabel('Spikes')
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.fill_between(time_ms, 0, I_input, alpha=0.5, color='g')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('I (nA)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig('01_single_neuron_voltage')
plt.close()

## Section 2: Population Models

Scale to a population of 100 neurons with E/I connectivity.

In [ ]:
def create_ei_connectivity(n_cells, sparsity=0.9):
    W = np.zeros((n_cells, n_cells))
    n_e = n_cells // 4
    e_idx = np.arange(n_e)
    i_idx = np.arange(n_e, n_cells)
    
    W[np.ix_(e_idx, e_idx)] = 1.0
    W[np.ix_(i_idx, e_idx)] = 2.0
    W[np.ix_(e_idx, i_idx)] = -1.0
    W[np.ix_(i_idx, i_idx)] = -1.0
    np.fill_diagonal(W, 0)
    
    mask = np.random.rand(n_cells, n_cells) < sparsity
    W[mask] = 0
    return W

n_pop = 100
W = create_ei_connectivity(n_pop)
print(f'Connectivity: {np.count_nonzero(W) / (n_pop**2):.1%} density')

In [ ]:
class PopulationModel:
    def __init__(self, n, W, dt=0.1):
        self.n, self.W, self.dt = n, W, dt
        self.v = np.ones(n) * -65
        self.u = np.ones(n) * -13
        self.spikes = np.zeros(n)
    
    def step(self, I_ext):
        I_syn = self.W @ self.spikes
        I = I_ext + I_syn
        dv = 0.04 * self.v**2 + 5 * self.v + 140 - self.u + I
        du = 0.02 * (0.2 * self.v - self.u)
        self.v += dv * self.dt
        self.u += du * self.dt
        self.spikes = (self.v >= 30).astype(float)
        self.v[self.spikes > 0] = -65
        self.u[self.spikes > 0] += 8
        return self.spikes.copy()

pop = PopulationModel(n_pop, W)
duration_ms = 1000
n_steps = int(duration_ms / 0.1)
time_ms = np.arange(n_steps) * 0.1

spikes_pop = np.zeros((n_pop, n_steps))
for i in range(n_steps):
    I_ext = np.zeros(n_pop)
    I_ext[:n_pop // 4] = 5
    spikes = pop.step(I_ext)
    spikes_pop[:, i] = spikes

rate = spikes_pop.sum() / (n_pop * duration_ms / 1000)
print(f'Population simulation: {spikes_pop.sum():.0f} spikes, {rate:.1f} Hz mean rate')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
for i in range(n_pop):
    times = time_ms[spikes_pop[i, :] > 0]
    ax.vlines(times, i - 0.4, i + 0.4, colors='k', linewidth=0.3)
ax.set_xlim(0, duration_ms)
ax.set_ylabel('Cell')
ax.set_title('Population Raster')
ax.grid(True, alpha=0.3)

ax = axes[1]
window = 50
rate_smooth = np.convolve(spikes_pop.sum(axis=0), np.ones(window)/window, mode='same')
ax.plot(time_ms, rate_smooth * (1000 / window), 'b-', linewidth=1)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Rate (Hz)')
ax.set_title('Population Rate')
ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig('02_population_raster')
plt.close()

## Section 3: Cortical Column

Build a laminar cortical structure with layers and cell types.

In [ ]:
def create_column(n_total=100):
    layers = ['L2/3', 'L4', 'L5', 'L6']
    depths = {'L2/3': (200, 400), 'L4': (400, 600), 'L5': (600, 800), 'L6': (800, 1000)}
    
    cells = []
    for layer in layers:
        n_layer = n_total // 4
        z_min, z_max = depths[layer]
        for i in range(n_layer):
            cell_type = np.random.choice(['E', 'PV', 'SST', 'VIP'], p=[0.65, 0.2, 0.1, 0.05])
            cells.append({
                'layer': layer,
                'cell_type': cell_type,
                'depth': np.random.uniform(z_min, z_max)
            })
    return pd.DataFrame(cells)

col_df = create_column(100)
print(col_df.groupby(['layer', 'cell_type']).size())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
colors = {'E': 'r', 'PV': 'b', 'SST': 'g', 'VIP': 'orange'}
sizes = {'E': 150, 'PV': 50, 'SST': 50, 'VIP': 50}

for cell_type in ['E', 'PV', 'SST', 'VIP']:
    subset = col_df[col_df['cell_type'] == cell_type]
    ax.scatter(np.zeros(len(subset)) + np.random.normal(0, 5, len(subset)), 
              subset['depth'], c=colors[cell_type], s=sizes[cell_type], 
              alpha=0.6, label=cell_type)

for z in [200, 400, 600, 800, 1000]:
    ax.axhline(z, color='k', linestyle='--', linewidth=1, alpha=0.3)

ax.set_xlim(-30, 30)
ax.set_ylim(1000, 100)
ax.set_ylabel('Depth (μm)')
ax.set_title('Cortical Column Layout')
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
savefig('03_cortical_column')
plt.close()

## Section 4: Optimization for Hypothesis Testing

Fine-tune L4 input to match target firing rates.

In [ ]:
def evaluate_column_tuning(l4_drive, col_df, seed=42):
    np.random.seed(seed)
    n = len(col_df)
    W = create_ei_connectivity(n, sparsity=0.9)
    pop = PopulationModel(n, W)
    
    duration_ms = 500
    n_steps = int(duration_ms / 0.1)
    
    spikes = np.zeros((n, n_steps))
    e_idx = col_df[col_df['cell_type'] == 'E'].index
    
    for i in range(n_steps):
        I_ext = np.zeros(n)
        I_ext[e_idx] = 3
        I_ext[:n // 4] = l4_drive  # L4 input
        spikes[:, i] = pop.step(I_ext)
    
    e_rate = spikes[e_idx].sum() / (len(e_idx) * duration_ms / 1000)
    return e_rate

targets = {'e_rate': 8}
best_loss, best_l4 = np.inf, 5
history = []

for i in range(50):
    l4 = np.random.uniform(2, 15)
    e_rate = evaluate_column_tuning(l4, col_df)
    loss = (e_rate - targets['e_rate'])**2
    history.append({'iter': i, 'l4': l4, 'e_rate': e_rate, 'loss': loss})
    if loss < best_loss:
        best_loss, best_l4 = loss, l4

print(f'Optimization: Best L4_drive={best_l4:.2f}, loss={best_loss:.2f}')

In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(hist_df['iter'], hist_df['loss'], 'b.-', alpha=0.7)
ax.axhline(best_loss, color='r', linestyle='--', label=f'Best: {best_loss:.2f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('Optimization Loss')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(hist_df['iter'], hist_df['e_rate'], 'g.-', alpha=0.7, label='E rate')
ax.axhline(targets['e_rate'], color='r', linestyle='--', label=f'Target: {targets["e_rate"]}')
ax.set_xlabel('Iteration')
ax.set_ylabel('E rate (Hz)')
ax.set_title('E Firing Rate Evolution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
savefig('04_optimization_history')
plt.close()

## Artifact Export

Save metrics and manifest.

In [ ]:
metrics = {
    'best_l4_drive': float(best_l4),
    'best_e_rate_hz': float(evaluate_column_tuning(best_l4, col_df)),
    'target_e_rate_hz': float(targets['e_rate']),
    'n_cells': len(col_df),
    'optimization_iterations': 50
}

pd.DataFrame([metrics]).to_csv('tutorial_metrics.csv', index=False)
hist_df.to_csv('tuning_history.csv', index=False)
print('✓ tutorial_metrics.csv')
print('✓ tuning_history.csv')

In [ ]:
manifest = {
    'tutorial_name': 'jaxfne Computational Biophysics Tutorial',
    'created_utc': datetime.utcnow().isoformat(),
    'truth_mode': 'truth_safe_unverified',
    'claim_level': 'computational_scaffold',
    'physical_amplitude_claim_allowed': False,
    'jaxfne_version': jaxfne_version,
    'execution_mode': 'tutorial_fallback',
    'source_calibration_status': 'toy_scale_not_empirical',
    'readout_status': 'proxy_only',
    'n_cells': len(col_df),
    'optimization_iterations': 50,
    'best_parameters': {'l4_drive': float(best_l4)},
    'best_metrics': metrics,
    'limitations': [
        'Izhikevich model, not biophysically calibrated',
        'No field solver, LFP/CSD are proxies only',
        'Optimization is random search, not production AGSDR',
        'No physical amplitude claims',
        'Tutorial scaffold only'
    ]
}

json_safe_dump(manifest, 'tutorial_manifest.json')

In [ ]:
print('\n' + '='*60)
print('TUTORIAL ARTIFACTS')
print('='*60)

figs = list(Path('figures').glob('*.png'))
csvs = list(Path('.').glob('*.csv'))
jsons = list(Path('.').glob('tutorial_manifest.json'))

print(f'\nFigures: {len(figs)} PNG + {len(list(Path("figures").glob("*.svg")))} SVG')
print(f'CSVs: {len(csvs)}')
print(f'Manifest: {len(jsons)}')
print('\nTutorial complete!')